In [ ]:
# =============================================================================
# [Step 1] 환경 설정 & 라이브러리 설치
# =============================================================================
print("⏳ 라이브러리 설치 및 설정 중...")
!pip install xgboost > /dev/null  # Colab에는 기본 설치되어 있으나 업데이트 권장
!pip install gdown > /dev/null
print("✅ 설치 완료.")

import gdown
import pandas as pd
import numpy as np
import math
import gc
import os
import shutil
import zipfile
from collections import defaultdict, Counter
from tqdm import tqdm
from xgboost import XGBClassifier # XGBoost 임포트

# =============================================================================
# [Step 2] 데이터 파일 준비 (기존과 동일)
# =============================================================================
zip_file_id = 'YOUR_DRIVE_FILE_ID'
target_zip = 'data.zip'
data_dir = 'data'

if not os.path.exists(target_zip):
    print(f"📥 데이터 다운로드 중...")
    url = f'https://drive.google.com/uc?id={zip_file_id}'
    gdown.download(url, target_zip, quiet=False)

if not os.path.exists(data_dir):
    os.makedirs(data_dir)

print("📦 압축 해제 중...")
with zipfile.ZipFile(target_zip, 'r') as zip_ref:
    zip_ref.extractall(data_dir)

required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
for filename in required_files:
    target_path = os.path.join(data_dir, filename)
    if not os.path.exists(target_path):
        for root, dirs, files in os.walk(data_dir):
            if filename in files:
                shutil.move(os.path.join(root, filename), target_path)
                break
        if not os.path.exists(target_path) and os.path.exists(filename):
            shutil.move(filename, target_path)

print(f"📂 데이터 준비 완료: {os.listdir(data_dir)}")

# =============================================================================
# [Step 3] 자모 유틸리티 (기존과 동일)
# =============================================================================
BASE_CODE = 0xAC00
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    if not isinstance(text, str): return []
    if len(text) > 100: text = text[:100]
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
        else:
            cho = code // 588
            jung = (code % 588) // 28
            jong = (code % 588) % 28
            result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            code = 0xAC00 + (CHOSUNG_LIST.index(cho)*588) + (JUNGSUNG_LIST.index(jung)*28) + JONGSUNG_LIST.index(jong)
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    MAX_LEN = 300
    if len(src_list) > MAX_LEN: src_list = src_list[:MAX_LEN]
    if len(tgt_list) > MAX_LEN: tgt_list = tgt_list[:MAX_LEN]
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    alignment = []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and dp[i][j] == dp[i-1][j] + 1:
            alignment.append((src_list[i-1], '<DEL>')); i-=1
        elif j>0 and dp[i][j] == dp[i][j-1] + 1:
            alignment.append((None, tgt_list[j-1])); j-=1
        else:
            alignment.append((src_list[i-1], tgt_list[j-1])); i-=1; j-=1
    return alignment[::-1]

# =============================================================================
# [Step 4] 디스크 기반 데이터 생성 (수정됨: y라벨 리매핑 추가)
# =============================================================================
def generate_dataset_disk_safe(csv_path, token_to_id, label_to_id, window_size=7):
    # token_to_id: 입력(X)를 위한 맵 (모든 자모)
    # label_to_id: 정답(y)를 위한 맵 (Target 자모만, 0부터 연속)

    PAD = '<PAD>'
    pad_id = token_to_id.get(PAD, 0)

    print("👀 1차 스캔: 샘플 개수 파악 중...")
    total_samples = 0
    for chunk in pd.read_csv(csv_path, chunksize=5000, encoding='utf-8-sig'):
        for _, row in chunk.iterrows():
            in_jamos = to_jamo_list(row['input'])
            out_jamos = to_jamo_list(row['output'])
            total_samples += max(len(in_jamos), len(out_jamos)) + 2
        del chunk
        gc.collect()

    print(f"📝 할당할 최대 샘플 수: {total_samples:,}")

    X_shape = (total_samples, window_size)
    y_shape = (total_samples,)

    if os.path.exists('X_data.dat'): os.remove('X_data.dat')
    if os.path.exists('y_data.dat'): os.remove('y_data.dat')

    X_mm = np.memmap('X_data.dat', dtype='float32', mode='w+', shape=X_shape)
    y_mm = np.memmap('y_data.dat', dtype='int32', mode='w+', shape=y_shape)

    print("✍️ 2차 스캔: 디스크에 데이터 기록 중...")
    current_idx = 0
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=2000, encoding='utf-8-sig'), desc="Disk Write"):
        for _, row in chunk.iterrows():
            in_jamos = to_jamo_list(row['input'])
            out_jamos = to_jamo_list(row['output'])
            alignment = align_jamo_list(in_jamos, out_jamos)

            in_ids = [token_to_id.get(t, 0) for t in in_jamos]
            padded_ids = [pad_id]*(window_size//2) + in_ids + [pad_id]*(window_size//2)

            curr = 0
            for src, tgt in alignment:
                if src is None: continue
                win = padded_ids[curr : curr + window_size]

                # [핵심 수정] 정답(y)은 label_to_id를 사용하여 0..K-1로 변환
                if tgt in label_to_id:
                    tgt_id = label_to_id[tgt]

                    if current_idx < total_samples:
                        X_mm[current_idx] = win
                        y_mm[current_idx] = tgt_id
                        current_idx += 1
                curr += 1

        if current_idx % 10000 == 0:
            X_mm.flush()

    print(f"✅ 데이터 생성 완료. 유효 샘플 수: {current_idx:,}")
    return X_mm[:current_idx], y_mm[:current_idx]

# =============================================================================
# [Step 5] XGBoost 학습 (Full Training)
# =============================================================================
print("🏗️ 토큰 맵 구축 중...")

# 1. 전체 토큰 맵 (Input용)
token_set = set(['<PAD>', '<DEL>'])
# 2. 타겟 토큰 맵 (Output용) - <DEL> 포함
target_set = set(['<DEL>'])

for chunk in pd.read_csv('data/train.csv', chunksize=5000, encoding='utf-8-sig'):
    text_blob = "".join(chunk['input'].astype(str).tolist()) + "".join(chunk['output'].astype(str).tolist())
    if len(text_blob) > 50000: text_blob = text_blob[:50000]
    token_set.update(to_jamo_list(text_blob))

    # 정답지용 자모 수집
    for txt in chunk['output'].astype(str):
        jamos = to_jamo_list(txt)
        target_set.update(jamos)

    del chunk, text_blob
    gc.collect()

# 입력용 맵 (전체)
token_to_id = {t: i for i, t in enumerate(sorted(list(token_set)))}
id_to_token = {i: t for t, i in token_to_id.items()} # 디버깅용

# [핵심] 정답용 맵 (구멍 없이 0부터 다시 매핑)
target_list = sorted(list(target_set))
label_to_id = {t: i for i, t in enumerate(target_list)}
id_to_label = {i: t for t, i in label_to_id.items()} # 추론 때 사용

print(f"✅ Input 토큰 수: {len(token_to_id)}")
print(f"✅ Output 클래스 수: {len(label_to_id)} (이 숫자가 XGBoost 클래스 수와 일치해야 함)")

# 데이터셋 로드 (y는 label_to_id로 변환됨)
X_disk, y_disk = generate_dataset_disk_safe('data/train.csv', token_to_id, label_to_id, window_size=7)
print(f"📊 총 학습 데이터: {len(X_disk):,}개")

print("🚀 XGBoost GPU 학습 시작 (전체 데이터 학습)...")

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42,
    n_jobs=-1,
    # 클래스 개수 명시 (안전장치)
    num_class=len(label_to_id),
    objective='multi:softprob'
)

xgb_model.fit(X_disk, y_disk)
print("✅ XGBoost 학습 완료!")

del X_disk, y_disk
gc.collect()
if os.path.exists('X_data.dat'): os.remove('X_data.dat')
if os.path.exists('y_data.dat'): os.remove('y_data.dat')

In [ ]:
# =============================================================================
# [Step 6] 추론 및 저장 (최적화: CuPy로 GPU 고속 추론)
# =============================================================================
import cupy as cp # GPU 배열 라이브러리

# 1. 혹시 메모리에서 사라졌을 경우를 대비해 LM 다시 로드
if 'lm_prob' not in locals():
    print("📚 언어 모델(LM) 로드 중...")
    lm_prob = build_language_model_stream('data/train.csv')

print(f"🚀 최종 추론 재시작 (업데이트 주기: 10개)...")
sample_submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')
outputs = []

# [핵심] 10개씩만 읽어서 처리 -> 게이지가 바로바로 움직임
test_chunk_iter = pd.read_csv('data/test.csv', chunksize=10, encoding='utf-8-sig')

# 전체 진행률 표시를 위한 계산
total_rows = 11263
total_chunks = total_rows // 10 + 1

for chunk in tqdm(test_chunk_iter, total=total_chunks, desc="실시간 복구 중"):
    # ------------------------------------------------------------------
    # 1. 배치 전처리 (한 번에 변환)
    # ------------------------------------------------------------------
    batch_X = []
    batch_lens = []

    pad_id = token_to_id.get('<PAD>', 0)

    for _, row in chunk.iterrows():
        text = row['input']
        if len(text) > 100: text = text[:100] # 길이 제한
        in_jamos = to_jamo_list(text)
        batch_lens.append(len(in_jamos))

        in_ids = [token_to_id.get(t, 0) for t in in_jamos]
        padded_ids = [pad_id]*(7//2) + in_ids + [pad_id]*(7//2)

        # 윈도우 슬라이딩
        for i in range(len(in_jamos)):
            batch_X.append(padded_ids[i : i + 7])

    if not batch_X:
        outputs.extend([""] * len(chunk))
        continue

    # ------------------------------------------------------------------
    # 2. GPU 배치 추론 (한 번에 예측)
    # ------------------------------------------------------------------
    X_gpu = cp.array(batch_X, dtype=cp.float32)
    probs_gpu = xgb_model.predict_proba(X_gpu) # 여기서 시간이 좀 걸림

    # CPU로 가져오기
    if hasattr(probs_gpu, 'get'):
        all_probs = probs_gpu.get()
    else:
        all_probs = probs_gpu

    # ------------------------------------------------------------------
    # 3. 결과 조립 (Beam Search)
    # ------------------------------------------------------------------
    cursor = 0
    model_classes = xgb_model.classes_

    for i, length in enumerate(batch_lens):
        # 해당 문장의 확률만 잘라내기
        curr_text_probs = all_probs[cursor : cursor + length]
        cursor += length

        # 빔 서치 (Beam Size = 5로 줄여서 속도 2배 향상)
        beam_size = 5
        beam = [([], None, 0.0)]

        for p_idx in range(len(curr_text_probs)):
            curr_probs = curr_text_probs[p_idx]
            top_k = np.argsort(curr_probs)[-beam_size:]

            new_beam = []
            for (decoded, last_char, score) in beam:
                for idx in top_k:
                    char_id = model_classes[int(idx)]
                    char_prob = float(curr_probs[idx])
                    if char_prob <= 0: char_prob = 1e-9

                    pred_char = id_to_label.get(char_id, '')

                    # 점수 계산
                    ml_score = math.log(char_prob)
                    lm_p = lm_prob.get((last_char, pred_char), 1e-7)
                    lm_score = math.log(lm_p)
                    total_score = score + ml_score + (2.5 * lm_score)

                    if pred_char == '<DEL>':
                        new_beam.append((decoded, last_char, total_score))
                    else:
                        new_beam.append((decoded + [pred_char], pred_char, total_score))

            new_beam.sort(key=lambda x: x[2], reverse=True)
            beam = new_beam[:beam_size]

        # 베스트 후보 선정
        candidates = []
        for (decoded, _, sc) in beam:
            word = from_jamo_list(decoded)
            candidates.append((word, sc))

        if not candidates:
            outputs.append("")
        else:
            best_word, best_score = candidates[0]
            # 사전 체크
            if best_word in vocab_set:
                outputs.append(best_word)
            else:
                # 차선책 검색
                found = False
                for j in range(1, len(candidates)):
                    cand_word, cand_score = candidates[j]
                    if best_score - cand_score > 3.0: break
                    if cand_word in vocab_set:
                        outputs.append(cand_word)
                        found = True
                        break
                if not found:
                    outputs.append(best_word)

    gc.collect()

# 저장
sample_submission['output'] = outputs
sample_submission.to_csv('submission_xgb_final.csv', index=False)
print(f"\n🎉 작업 완료! 총 {len(outputs)}개 복구 성공.")